# 01 - Preprocessing

In [ ]:
%autosave 0

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import rapids_singlecell as rsc
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import marsilea as ma
import marsilea.plotter as mp
import plotting.settings
from dynaconf import Dynaconf

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Load data 

In [ ]:
# Read the count matrix
adata = sc.read_mtx(data_dir / "count_matrix.mtx")

In [ ]:
genes = pd.read_csv(data_dir / "all_genes.csv")
cell_meta = pd.read_csv(data_dir / "cell_metadata.csv", index_col=0)

In [ ]:
adata.var = genes.set_index("gene_name")
adata.obs = cell_meta

In [ ]:
# Write anndata object to file
anndata_dir.mkdir(parents=True, exist_ok=True)
adata.write(anndata_dir / "tf_ko_panel_combined_raw.h5ad")

## QC

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_combined_raw.h5ad")

In [ ]:
# Compute QC metrics
adata.var["mito"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mito"], inplace=True)

In [ ]:
# Since we want to use gene symbols, we have to make sure that var_names are unique
adata.var_names_make_unique()

In [ ]:
# First plot genes that yield highest fraction of counts
sc.pl.highest_expr_genes(adata, n_top=20)

In [ ]:
# Plot QC metrics across samples
with mpl.rc_context(rc={'figure.figsize': (20, 4)}):
    for key in ["n_genes_by_counts", "total_counts", "pct_counts_mito"]:
        fig, ax = plt.subplots()
        sc.pl.violin(
            adata,
            keys=key,
            groupby="sample",
            jitter=0.4,          # scatter points per violin
            rotation=90,         # rotate x-axis labels
            show=False,           # show in notebook
            ax=ax,
        )
        if key=="n_genes_by_counts":
            ax.axhline(200, color='red', linestyle='--')
        plt.tight_layout()
        plt.savefig(sc.settings.figdir / f"violin_{key}_by_sample_before_filtering.pdf", dpi=300)

In [ ]:
sc.pp.filter_cells(adata, min_counts=500)
sc.pp.filter_cells(adata, max_counts=100000)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs.pct_counts_mito <= 40, :]

In [ ]:
adata

In [ ]:
# Plot QC metrics across samples
with mpl.rc_context(rc={'figure.figsize': (20, 4)}):
    for key in ["n_genes_by_counts", "total_counts", "pct_counts_mito"]:
        fig, ax = plt.subplots()
        sc.pl.violin(
            adata,
            keys=key,
            groupby="sample",
            jitter=0.4,          # scatter points per violin
            rotation=90,         # rotate x-axis labels
            show=False,           # show in notebook
            ax=ax,
        )
        plt.tight_layout()
        plt.savefig(sc.settings.figdir / f"violin_{key}_by_sample_after_filtering.pdf", dpi=300)

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_combined_filtered.h5ad")

### Plot how many cells per sample we have


In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_combined_filtered.h5ad")

In [ ]:
adata.shape

In [ ]:
adata.layers['counts'] = adata.X
adata.raw = adata.copy()

In [ ]:
cells_per_sample = adata.obs.groupby(["sample"]).size().sort_values(ascending=False)
mean_cells_per_sample = int(cells_per_sample.mean())

fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(cells_per_sample.index, cells_per_sample.values, color='grey')
ax.tick_params(axis='x', rotation=90)
ax.set_ylabel("Number of cells")
ax.axhline(mean_cells_per_sample, c='red', linestyle='--', label=f'Mean cells per sample: {mean_cells_per_sample}')
ax.grid(False)
ax.legend()
ax.set_xlim(-0.5, len(cells_per_sample) - 0.5)
fig.tight_layout()
fig.show()
fig.savefig(sc.settings.figdir / f"bar_cells_by_sample.pdf", dpi=300)

### Downstream processing

In [ ]:
# Library size normalization
sc.pp.normalize_total(adata, target_sum=1e4)

# Log-transform counts
sc.pp.log1p(adata)

# Store the log-normalized data in a separate layer
adata.layers["lognorm"] = adata.X.copy()

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    flavor="seurat_v3",
    n_top_genes=5000,
    batch_key="sample"  
)

In [ ]:
sc.pl.highly_variable_genes(adata)

In [ ]:
# Keep only HVGs
adata = adata[:, adata.var["highly_variable"]].copy()

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_combined_hvg.h5ad")

## Downstream processing

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_combined_hvg.h5ad")

In [ ]:
rsc.get.anndata_to_GPU(adata)

In [ ]:
rsc.pp.regress_out(adata, keys=["total_counts", "pct_counts_mito"])

In [ ]:
rsc.pp.scale(adata, max_value=10)

In [ ]:
rsc.pp.pca(adata, n_comps=100, use_highly_variable=False)

In [ ]:
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=100)

In [ ]:
rsc.pp.neighbors(adata, n_neighbors=15, n_pcs=50)
rsc.tl.umap(adata)

In [ ]:
for resolution in [0.3, 0.5, 1.0, 1.5, 2.0]:
    rsc.tl.leiden(adata, resolution=resolution, key_added=f"pca_leiden_{resolution}")  

In [ ]:
del adata.uns["log1p"]
del adata.uns["pca"]

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_combined_processed.h5ad")

## Export the processed anndata object

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_combined_processed.h5ad")

In [ ]:
%%time
rsc.tl.rank_genes_groups_logreg(adata, groupby="pca_leiden_2.0", use_raw=False)

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
del adata.uns["log1p"]

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_combined_processed_vis.h5ad")

In [ ]:
adata

In [ ]:
dedf = pd.concat([sc.get.rank_genes_groups_df(adata, group=group).assign(group=group) for group in list(adata.obs["pca_leiden_2.0"].unique())], axis=0)

In [ ]:
out_dir = Path(settings.ANALYSIS.tables_dir)
out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
dedf.to_csv(out_dir / "markers.csv")